# 实践 9.1  组学矩阵的QC、缺失值与批次效应

《统计遗传与智能育种学》第九章 · 电子资源 `09_multiomics_qc.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：09_multiomics_qc.ipynb。实践生成一个带处理效应、仪器漂移、批次偏移和低丰度缺失的模拟丰度矩阵。代码完成缺失统计、log转换、池化QC漂移诊断和PCA结构诊断。预测部分明确比较两条流程：错误流程在全数据上拟合插补/尺度并用全体标签选特征；训练集限定流程只在训练集拟合插补、尺度和监督式特征选择，再独立评价测试集。

运行检查：重启内核并运行全部单元  
预期：输出缺失率、池化QC随上机顺序漂移的相关、PCA结构诊断；固定种子下泄漏流程测试准确率约0.738，训练集限定流程约0.690。该差值仅用于说明泄漏可能造成乐观偏差，不是普遍的偏差幅度。

## 输入、完整代码、实际输出与解释

输入与可复现条件：含处理效应、仪器漂移、批次偏移和低丰度缺失的模拟丰度矩阵；SEED=901；预测任务固定独立测试集。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=901
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 901


In [2]:
from sklearn.decomposition import PCA
ns,nf=120,90; treatment=np.tile([0,1],ns//2); batch=np.repeat([0,1,2],40); run=np.arange(ns); rng.shuffle(run); base=rng.lognormal(2,.7,(ns,nf)); effect=np.ones(nf); effect[:12]=1.45; X=base*effect[None,:]**treatment[:,None]; drift_coef=rng.normal(0,.002,nf); X*=np.exp((run-run.mean())[:,None]*drift_coef); X*=np.exp(batch[:,None]*rng.normal(0,.08,nf)); miss_prob=1/(1+np.exp((np.log1p(X)-1.8)*3)); miss=rng.random(X.shape)<.12*miss_prob; X[miss]=np.nan; print('sample missing range',np.round([np.mean(np.isnan(X),1).min(),np.mean(np.isnan(X),1).max()],3)); print('feature missing median',round(float(np.median(np.mean(np.isnan(X),0))),3)); qc_run=np.linspace(run.min(),run.max(),12); qc_ref=np.nanmedian(base,axis=0); QC=qc_ref[None,:]*np.exp((qc_run-qc_run.mean())[:,None]*drift_coef); qc_signal=np.log2(QC+1).mean(1); print('pooled-QC mean signal vs run r',round(np.corrcoef(qc_signal,qc_run)[0,1],3))

# log转换 + 中位数填补用于QC演示；真实项目应按数据类型选择归一化/缺失模型
L=np.log2(X+1); med=np.nanmedian(L,axis=0); ii=np.where(np.isnan(L)); L[ii]=med[ii[1]]; pcs=PCA(3).fit_transform(L); print('PCA PC1 correlations: batch',round(np.corrcoef(pcs[:,0],batch)[0,1],3),'run',round(np.corrcoef(pcs[:,0],run)[0,1],3),'treatment',round(np.corrcoef(pcs[:,0],treatment)[0,1],3))

# 预测评价：全数据标签选特征会泄漏；干净流程只用训练集拟合插补、尺度和特征选择。
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
tr,te=train_test_split(np.arange(ns),test_size=.35,stratify=treatment,random_state=SEED)
raw=np.log2(X+1)
# 错误示范：先用全数据拟合插补/尺度，并用全数据标签选特征。
imp_all=SimpleImputer(strategy='median').fit(raw); Z_all=imp_all.transform(raw); sc_all=StandardScaler().fit(Z_all); Z_all=sc_all.transform(Z_all)
sel_leak=np.argsort([abs(np.corrcoef(Z_all[:,j],treatment)[0,1]) for j in range(nf)])[-12:]
m_leak=LogisticRegression(max_iter=1000).fit(Z_all[tr][:,sel_leak],treatment[tr])
print('leaky full-data preprocessing/selection accuracy',round(m_leak.score(Z_all[te][:,sel_leak],treatment[te]),3))
# 正确示范：所有可学习的预处理参数和监督式特征选择只在训练集拟合。
imp=SimpleImputer(strategy='median').fit(raw[tr]); Ztr=imp.transform(raw[tr]); Zte=imp.transform(raw[te]); sc=StandardScaler().fit(Ztr); Ztr=sc.transform(Ztr); Zte=sc.transform(Zte)
sel_train=np.argsort([abs(np.corrcoef(Ztr[:,j],treatment[tr])[0,1]) for j in range(nf)])[-12:]
m=LogisticRegression(max_iter=1000).fit(Ztr[:,sel_train],treatment[tr]); print('train-only preprocessing/selection accuracy',round(m.score(Zte[:,sel_train],treatment[te]),3))


sample missing range [0.  0.1]
feature missing median 0.042
pooled-QC mean signal vs run r -0.999
PCA PC1 correlations: batch 0.361 run 0.13 treatment -0.34
leaky full-data preprocessing/selection accuracy 0.738
train-only preprocessing/selection accuracy 0.69


结果解释：池化QC平均信号与上机顺序相关−0.999，PC1同时与batch、run和treatment相关，说明降维轴不能自动等同某个生物过程。错误的全数据预处理/监督选特征得到测试准确率0.738，而训练集限定流程为0.690；差值说明泄漏可制造乐观结果，但0.048不是可外推的固定偏差。

可修改练习：依次将插补、尺度拟合和监督选特征限制在训练集，分别量化性能变化；重复多个SEED观察偏差分布。